[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/40_mlops/04_deployment_strategies.ipynb)

# Bài 04 — Chiến lược triển khai: shadow, canary, A/B test

**🎯 Mục tiêu**
- So sánh các chiến lược đưa mô hình mới vào production: thay thẳng, blue-green, **shadow**, **canary**, **A/B test**, bandit.
- Tự viết **bộ định tuyến** (router) chia traffic ổn định theo người dùng; chạy shadow để so hai mô hình **không ảnh hưởng người dùng**.
- Mô phỏng **canary có tự động rollback** dựa trên kiểm định thống kê.
- Thiết kế A/B test: **cỡ mẫu** theo power, kiểm chứng bằng mô phỏng; thấy tác hại của **"nhìn trộm"** (peeking); bandit Thompson sampling.
- Nhận diện **vòng phản hồi** (feedback loop): hành động của mô hình làm hỏng dữ liệu huấn luyện của chính nó — và cách phòng bằng nhóm đối chứng.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/40_mlops

In [ ]:
import hashlib
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from ml_course.datasets import make_customer_churn

FEATURES = ["tenure_months", "contract", "internet_service", "monthly_charges", "total_charges",
            "payment_method", "support_calls", "senior", "n_products"]
NUMERIC = ["tenure_months", "monthly_charges", "total_charges", "support_calls", "n_products"]
CATEGORICAL = ["contract", "internet_service", "payment_method", "senior"]


def make_model(estimator):
    return make_pipeline(ColumnTransformer([("num", StandardScaler(), NUMERIC), ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL)]),
                         estimator)

## 1. Các chiến lược

| Chiến lược | Cách làm | Rủi ro cho người dùng | Đo được gì | Khi nào dùng |
|---|---|---|---|---|
| Thay thẳng (recreate) | tắt bản cũ, bật bản mới | cao — lỗi ảnh hưởng 100% | — | môi trường thử nghiệm |
| Blue-green | chạy song song 2 môi trường, chuyển toàn bộ traffic một lần | trung bình — rollback tức thì bằng chuyển lại | — | cần rollback nhanh |
| **Shadow** | bản mới nhận **bản sao** traffic, dự đoán chỉ được log, không dùng | **không** | độ trễ, lỗi, phân phối dự đoán, mức đồng thuận | luôn nên làm trước tiên |
| **Canary** | bản mới nhận 1% → 5% → 25% → 100% traffic, dừng nếu xấu | thấp, có kiểm soát | lỗi, độ trễ, metric sớm | phát hành phần mềm/mô hình |
| **A/B test** | chia ngẫu nhiên người dùng, chạy đủ cỡ mẫu | có — một nửa dùng bản kém hơn trong lúc test | **tác động nhân quả** lên metric kinh doanh | quyết định giữ mô hình nào |
| Bandit | tự dồn traffic về bản đang tốt hơn | thấp hơn A/B | tối ưu phần thưởng, suy luận kém hơn | nhiều phương án, tối ưu ngắn hạn |

Metric offline (AUC) không đảm bảo metric kinh doanh tốt hơn: mô hình mới có thể chọn khách khác, gây phản ứng khác. Chỉ thí nghiệm online trả lời "mô hình mới có **làm tăng** tỉ lệ giữ chân không" (chương 38: nhân quả).

## 2. Bộ định tuyến traffic

Yêu cầu: (1) tỉ lệ traffic đúng như cấu hình; (2) **ổn định** — cùng một người dùng luôn vào cùng phiên bản (trải nghiệm nhất quán, và A/B test hợp lệ); (3) độc lập giữa các thí nghiệm (đổi "muối" — salt). Cách chuẩn: **băm** `salt + user_id` thành số trong [0, 1).

In [ ]:
def bucket(user_id: str, salt: str) -> float:
    """Ánh xạ ổn định (user, thí nghiệm) → [0, 1)."""
    digest = hashlib.sha256(f"{salt}:{user_id}".encode()).hexdigest()
    return int(digest[:15], 16) / 16 ** 15


def assign(user_id: str, weights: dict, salt: str = "churn-model-v2") -> str:
    u, cumulative = bucket(user_id, salt), 0.0
    for variant, w in weights.items():
        cumulative += w
        if u < cumulative:
            return variant
    return variant


users = [f"U{i:06d}" for i in range(100_000)]
split = pd.Series([assign(u, {"v1": 0.9, "v2": 0.1}) for u in users]).value_counts(normalize=True)
print("tỉ lệ thực tế:", split.round(4).to_dict())
print("ổn định — gán lại 1.000 người dùng cho kết quả y hệt:", all(assign(u, {"v1": 0.9, "v2": 0.1}) == assign(u, {"v1": 0.9, "v2": 0.1}) for u in users[:1000]))
grow = np.mean([assign(u, {"v2": 0.1, "v1": 0.9}) == "v2" and assign(u, {"v2": 0.25, "v1": 0.75}) == "v2" for u in users]) / 0.1
print(f"tăng canary 10% → 25%: {grow:.0%} người dùng đang ở v2 vẫn ở v2 (đặt v2 đứng đầu → các bucket lồng nhau)")
other = np.mean([assign(u, {"v1": 0.9, "v2": 0.1}) == "v2" for u in users if assign(u, {"A": 0.5, "B": 0.5}, salt="pricing-test") == "A"])
print(f"độc lập giữa hai thí nghiệm (salt khác): trong nhóm A của thí nghiệm giá, {other:.1%} vào v2 (≈ 10%)")

## 3. Shadow deployment

Mô hình mới (v2, gradient boosting) chạy **song song** với v1 trên cùng traffic; chỉ v1 quyết định. Ta so sánh trước khi để v2 ảnh hưởng bất kỳ ai.

In [ ]:
history = make_customer_churn(8000, seed=0)
v1 = make_model(LogisticRegression(max_iter=1000)).fit(history[FEATURES], history.churn)
v2 = make_model(HistGradientBoostingClassifier(random_state=0)).fit(history[FEATURES], history.churn)

traffic = make_customer_churn(3000, seed=42)                           # một ngày traffic thật
latency = {"v1": [], "v2": []}
for i in range(300):                                                   # đo độ trễ từng request trên 300 request đầu
    row = traffic.iloc[[i]][FEATURES]
    for name, m in [("v1", v1), ("v2", v2)]:                            # v1 trả kết quả; v2 chỉ log (thực tế chạy bất đồng bộ)
        t0 = time.perf_counter()
        m.predict_proba(row)
        latency[name].append((time.perf_counter() - t0) * 1000)
shadow = pd.DataFrame({"p_v1": v1.predict_proba(traffic[FEATURES])[:, 1], "p_v2": v2.predict_proba(traffic[FEATURES])[:, 1]})
top_v1, top_v2 = shadow.p_v1 >= shadow.p_v1.quantile(0.8), shadow.p_v2 >= shadow.p_v2.quantile(0.8)
print(f"tương quan dự đoán v1–v2: {shadow.p_v1.corr(shadow.p_v2):.3f} | trùng nhau trong top 20% được gọi giữ chân: {(top_v1 & top_v2).sum() / top_v1.sum():.0%}")
print(f"xác suất TB: v1 {shadow.p_v1.mean():.3f} · v2 {shadow.p_v2.mean():.3f} | độ trễ p99: v1 {np.percentile(latency['v1'], 99):.2f} ms · v2 {np.percentile(latency['v2'], 99):.2f} ms")
print(f"(nhãn thật về sau 30 ngày) AUC: v1 {roc_auc_score(traffic.churn, shadow.p_v1):.3f} · v2 {roc_auc_score(traffic.churn, shadow.p_v2):.3f}")

Shadow trả lời các câu hỏi **kỹ thuật** (v2 có lỗi không, đủ nhanh không, dự đoán có hợp lý không) và cho thấy dù dự đoán tương quan cao, v2 sẽ **chọn khác** v1 khoảng một phần sáu số khách được gọi. Nó **không** trả lời được câu hỏi kinh doanh — vì không ai hành động theo v2. Ở đây nhãn về còn cho thấy v2 không tốt hơn v1 (dữ liệu sinh từ mô hình logistic — chương 39).

## 4. Canary có tự động rollback

Phiên bản mới (ví dụ thêm tiền xử lý) có lỗi ẩn: với ~3% request (khách có `total_charges = 0`), nó ném ngoại lệ → HTTP 500. Bản cũ có tỉ lệ lỗi nền 0.2%. Canary tăng dần traffic và **tự rollback** khi tỉ lệ lỗi của canary cao hơn bản cũ **có ý nghĩa thống kê** (kiểm định hai tỉ lệ một phía), sau khi có đủ request tối thiểu.

In [ ]:
def error_rate_worse(err_new, n_new, err_old, n_old, alpha=0.01, min_requests=500):
    """True nếu tỉ lệ lỗi bản mới lớn hơn bản cũ có ý nghĩa (z-test một phía)."""
    if n_new < min_requests:
        return False
    p_new, p_old = err_new / n_new, err_old / n_old
    pooled = (err_new + err_old) / (n_new + n_old)
    se = np.sqrt(pooled * (1 - pooled) * (1 / n_new + 1 / n_old))
    return se > 0 and stats.norm.sf((p_new - p_old) / se) < alpha


def run_canary(new_error_rate, stages=(0.01, 0.05, 0.25, 0.5, 1.0), requests_per_stage=20_000, old_error_rate=0.002, seed=0):
    rng = np.random.default_rng(seed)
    err_new = n_new = err_old = n_old = 0
    for share in stages:
        k = rng.binomial(requests_per_stage, share)                    # số request vào canary ở giai đoạn này
        err_new += rng.binomial(k, new_error_rate)
        n_new += k
        err_old += rng.binomial(requests_per_stage - k, old_error_rate)
        n_old += requests_per_stage - k
        status = "ROLLBACK ⛔" if error_rate_worse(err_new, n_new, err_old, n_old) else "tiếp tục ✅"
        print(f"   canary {share:4.0%}: {n_new:6,} request, lỗi {err_new / max(n_new, 1):.2%} (bản cũ {err_old / n_old:.2%}) → {status}")
        if status.startswith("ROLLBACK"):
            return f"rollback ở mức {share:.0%} — chỉ {n_new:,} request bị ảnh hưởng"
    return "hoàn tất — bản mới nhận 100% traffic"


print("Bản mới có lỗi (3% request lỗi):")
print("  →", run_canary(new_error_rate=0.002 + 0.03))
print("Bản mới tốt (cùng tỉ lệ lỗi nền):")
print("  →", run_canary(new_error_rate=0.002))

- Bản lỗi bị chặn khi mới nhận phần nhỏ traffic, thay vì làm hỏng trải nghiệm của mọi người dùng.
- `min_requests` và mức ý nghĩa nghiêm (α = 0.01) tránh rollback nhầm do nhiễu khi canary còn ít request; nhưng vì kiểm tra **lặp lại** ở mỗi giai đoạn, xác suất báo động nhầm tích luỹ lớn hơn α (xem "nhìn trộm" ở phần 6).
- Thực tế còn gate theo độ trễ p99, CPU/bộ nhớ, và metric nghiệp vụ sớm. Công cụ: Argo Rollouts, Flagger, Istio, AWS CodeDeploy.

## 5. A/B test: bao nhiêu người dùng là đủ?

Mục tiêu kinh doanh: chiến dịch giữ chân dựa trên mô hình B có tăng tỉ lệ **giữ chân thành công** từ 10% lên 11% không? Cỡ mẫu mỗi nhóm để phát hiện chênh lệch $\delta = p_2 - p_1$ với mức ý nghĩa $\alpha$ (hai phía) và power $1 - \beta$:

$$n = \frac{\left(z_{1-\alpha/2}\sqrt{2\bar p(1-\bar p)} + z_{1-\beta}\sqrt{p_1(1-p_1) + p_2(1-p_2)}\right)^2}{\delta^2}$$

In [ ]:
def sample_size(p1, p2, alpha=0.05, power=0.8):
    p_bar = (p1 + p2) / 2
    num = stats.norm.ppf(1 - alpha / 2) * np.sqrt(2 * p_bar * (1 - p_bar)) + stats.norm.ppf(power) * np.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
    return int(np.ceil(num ** 2 / (p2 - p1) ** 2))


def two_proportion_p(x1, n1, x2, n2):
    pooled = (x1 + x2) / (n1 + n2)
    z = (x2 / n2 - x1 / n1) / np.sqrt(pooled * (1 - pooled) * (1 / n1 + 1 / n2))
    return 2 * stats.norm.sf(np.abs(z))


for p2 in [0.11, 0.12, 0.15]:
    print(f"10% → {p2:.0%}: cần {sample_size(0.10, p2):,} người mỗi nhóm")
n = sample_size(0.10, 0.11)
rng = np.random.default_rng(1)
x1, x2 = rng.binomial(n, 0.10, 4000), rng.binomial(n, 0.11, 4000)
x_null = rng.binomial(n, 0.10, 4000)
print(f"\nmô phỏng 4.000 thí nghiệm với n = {n:,}: power thực tế = {np.mean(two_proportion_p(x1, n, x2, n) < 0.05):.3f} (mục tiêu 0.80)"
      f" | tỉ lệ dương tính giả khi không có khác biệt = {np.mean(two_proportion_p(x1, n, x_null, n) < 0.05):.3f} (mục tiêu 0.05)")

Cải thiện tuyệt đối 1 điểm % cần hàng chục nghìn người mỗi nhóm — cỡ mẫu tỉ lệ nghịch với **bình phương** hiệu ứng. Vì vậy: chọn metric nhạy (gần với tác động của mô hình), giảm phương sai (CUPED — chương 38), và **quyết định cỡ mẫu trước** khi chạy.

## 6. "Nhìn trộm" kết quả (peeking)

Thói quen nguy hiểm: xem p-value **mỗi ngày** và dừng ngay khi p < 0.05. Mô phỏng **A/A test** (hai nhóm **giống hệt**, không có khác biệt thật) trong 20 ngày, mỗi ngày 500 người/nhóm:

In [ ]:
rng = np.random.default_rng(0)
n_sims, days, per_day = 4000, 20, 500
conv_a = rng.binomial(per_day, 0.10, (n_sims, days)).cumsum(axis=1)
conv_b = rng.binomial(per_day, 0.10, (n_sims, days)).cumsum(axis=1)
n_cum = per_day * np.arange(1, days + 1)
p_daily = two_proportion_p(conv_a, n_cum, conv_b, n_cum)               # p-value tại cuối mỗi ngày
fixed_horizon = np.mean(p_daily[:, -1] < 0.05)
peeking = np.mean((p_daily < 0.05).any(axis=1))
print(f"chỉ kiểm định một lần ở ngày 20: dương tính giả = {fixed_horizon:.1%}")
print(f"nhìn mỗi ngày, dừng khi p < 0.05: dương tính giả = {peeking:.1%}")
bonferroni = np.mean((p_daily < 0.05 / days).any(axis=1))
print(f"nhìn mỗi ngày với ngưỡng Bonferroni 0.05/20: dương tính giả = {bonferroni:.1%} (an toàn nhưng mất power)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for i in range(6):
    ax.plot(range(1, days + 1), p_daily[i], lw=1.2)
ax.axhline(0.05, color="red", ls="--", label="p = 0.05")
ax.set_yscale("log")
ax.set_xlabel("ngày")
ax.set_ylabel("p-value (thang log)")
ax.set_title("A/A test: p-value dao động ngẫu nhiên — đủ lâu sẽ có lúc chạm 0.05")
ax.legend()
plt.tight_layout()
plt.show()

Khi không có khác biệt thật, p-value là **bước ngẫu nhiên**; nhìn đủ nhiều lần, thế nào cũng có lúc nó dưới 0.05. Giải pháp: cố định cỡ mẫu/thời gian trước; hoặc dùng **kiểm định tuần tự** được thiết kế cho việc nhìn liên tục (alpha spending — O'Brien–Fleming, always-valid p-values / mSPRT) — các nền tảng thí nghiệm lớn (Optimizely, Statsig, Eppo) đều dùng.

## 7. Bandit: tối ưu trong lúc học

A/B test chia đều traffic suốt thời gian thí nghiệm — "trả giá" bằng người dùng ở nhánh kém. **Thompson sampling** (chương 35) dồn dần traffic về nhánh đang tốt hơn: mỗi người dùng, lấy mẫu tỉ lệ thành công của mỗi nhánh từ phân phối Beta hậu nghiệm, chọn nhánh có mẫu lớn nhất.

In [ ]:
def simulate_policy(policy, true_rates=(0.10, 0.13), n_users=20_000, seed=0):
    rng = np.random.default_rng(seed)
    wins, plays, choices = np.zeros(2), np.zeros(2), []
    for t in range(n_users):
        arm = t % 2 if policy == "A/B 50-50" else int(np.argmax(rng.beta(1 + wins, 1 + plays - wins)))
        reward = rng.random() < true_rates[arm]
        wins[arm] += reward
        plays[arm] += 1
        choices.append(arm)
    regret = n_users * max(true_rates) - sum(plays[a] * true_rates[a] for a in range(2))
    return wins.sum(), plays, regret, np.array(choices)


for policy in ["A/B 50-50", "Thompson sampling"]:
    total, plays, regret, choices = simulate_policy(policy)
    print(f"{policy:18s}: thành công = {total:,.0f} | người dùng ở nhánh kém = {plays[0]:,.0f} | regret kỳ vọng ≈ {regret:,.0f} | 2.000 người cuối vào nhánh tốt: {choices[-2000:].mean():.0%}")

Bandit mất ít người dùng hơn ở nhánh kém, nhưng: dữ liệu không cân bằng → **ước lượng hiệu ứng kém chính xác** hơn và khó kiểm định; giả định tỉ lệ không đổi theo thời gian (vi phạm khi có mùa vụ). Dùng bandit khi mục tiêu là **tối ưu** (chọn tiêu đề, gợi ý), dùng A/B khi cần **kết luận** có kiểm soát.

## 8. Vòng phản hồi: mô hình làm hỏng dữ liệu của chính nó

Mỗi tháng, mô hình chọn 30% khách rủi ro cao nhất để gọi giữ chân; cuộc gọi giảm 70% xác suất rời bỏ. Cuối tháng, ta **huấn luyện lại trên dữ liệu vừa thu thập**. Vấn đề: khách rủi ro cao **được gọi** nên ít rời bỏ → dữ liệu mới "dạy" rằng các dấu hiệu rủi ro không còn nguy hiểm (đây chính là trôi khái niệm do chính hệ thống gây ra — bài 03).

Hai cách huấn luyện lại:
- **Ngây thơ**: dùng mọi khách hàng của tháng.
- **Nhóm đối chứng**: giữ ngẫu nhiên 10% khách **không bao giờ** được gọi (holdout), chỉ huấn luyện trên nhóm này — dữ liệu phản ánh rủi ro **khi không can thiệp**.

In [ ]:
def churn_logit(df):
    return (-1.6 + 1.3 * (df.contract == "month-to-month") - 0.5 * (df.contract == "two_year") - 0.035 * df.tenure_months
            + 0.018 * (df.monthly_charges - 70) + 0.38 * df.support_calls + 0.45 * (df.internet_service == "fiber")
            + 0.35 * (df.payment_method == "e-wallet") + 0.3 * df.senior - 0.2 * (df.n_products - 1))


evaluation = make_customer_churn(20000, seed=999)                      # khách mới, đo khả năng xếp hạng RỦI RO KHI KHÔNG CAN THIỆP
initial = make_model(LogisticRegression(max_iter=1000)).fit(history[FEATURES], history.churn)
models = {"ngây thơ": initial, "nhóm đối chứng 10%": initial}
loop_rows = [{"tháng": 0, **{name: roc_auc_score(evaluation.churn, initial.predict_proba(evaluation[FEATURES])[:, 1]) for name in models}}]
for month in range(1, 7):
    customers = make_customer_churn(5000, seed=100 + month).drop(columns="churn")
    rng = np.random.default_rng(200 + month)
    holdout = rng.random(len(customers)) < 0.10
    outcome_noise = rng.random(len(customers))
    row = {"tháng": month}
    for name, m in models.items():
        score = m.predict_proba(customers[FEATURES])[:, 1]
        eligible = ~holdout if name.startswith("nhóm") else np.ones(len(customers), bool)
        called = (score >= np.quantile(score[eligible], 0.7)) & eligible
        p_churn = 1 / (1 + np.exp(-churn_logit(customers))) * np.where(called, 0.3, 1.0)
        observed = customers.assign(churn=(outcome_noise < p_churn).astype(int))
        train = observed if name == "ngây thơ" else observed[holdout]
        models[name] = make_model(LogisticRegression(max_iter=1000)).fit(train[FEATURES], train.churn)
        row[name] = roc_auc_score(evaluation.churn, models[name].predict_proba(evaluation[FEATURES])[:, 1])
    loop_rows.append(row)
print("AUC xếp hạng rủi ro thật (khi không can thiệp) qua các tháng huấn luyện lại:")
print(pd.DataFrame(loop_rows).set_index("tháng").round(3).to_string())

In [ ]:
coef = pd.DataFrame({name: pd.Series(m[-1].coef_[0], index=m[0].get_feature_names_out()) for name, m in [("ban đầu", initial)] + list(models.items())})
print("hệ số đã học (thang chuẩn hoá) của vài đặc trưng rủi ro:")
print(coef.loc[["num__support_calls", "num__tenure_months", "cat__contract_month-to-month"]].round(2).to_string())

- Mô hình **ngây thơ** xếp hạng rủi ro thật ngày càng kém: hệ số của các dấu hiệu rủi ro bị **teo** lại (hợp đồng tháng gần như mất tác dụng) vì chính những khách đó đã được cứu. Nó học "điều gì xảy ra **dưới chính sách hiện tại**", không phải "rủi ro nếu không làm gì".
- **Nhóm đối chứng** giữ được khả năng xếp hạng (dù chỉ huấn luyện trên 10% dữ liệu) và còn cho phép đo **tác động thật** của chiến dịch (so nhóm được gọi với nhóm đối chứng tương ứng). Chi phí: 10% khách rủi ro cao không được giúp — đánh đổi giữa khai thác và đo lường.
- Vòng phản hồi phổ biến: gợi ý (chỉ thấy phản hồi trên món đã gợi ý → thiên lệch phổ biến — chương 26), tín dụng (chỉ biết người **được duyệt** có trả nợ không — selective labels), phát hiện gian lận (giao dịch bị chặn không có nhãn), cảnh sát dự đoán. Cách xử lý: nhóm đối chứng/khám phá ngẫu nhiên, ghi lại **hành động và xác suất hành động** để hiệu chỉnh (IPW — chương 38), mô hình hoá tác động (uplift).

## 🧠 Kiểm tra nhanh

1. Shadow deployment trả lời được câu hỏi gì, không trả lời được câu hỏi gì?
2. Vì sao gán người dùng vào phiên bản phải dựa trên băm, không ngẫu nhiên mỗi request?
3. Canary tự động rollback dựa trên tiêu chí gì? Vì sao cần số request tối thiểu?
4. Cỡ mẫu A/B test phụ thuộc hiệu ứng thế nào? Muốn phát hiện hiệu ứng nhỏ bằng một nửa cần bao nhiêu mẫu?
5. Vì sao "nhìn trộm" p-value làm tăng dương tính giả? Cách xử lý?
6. Khi nào dùng bandit, khi nào dùng A/B test?
7. Vòng phản hồi là gì? Cho ví dụ và cách phòng.

## 📌 Tóm tắt

- Shadow trước (kỹ thuật, không rủi ro) → canary (an toàn, tự rollback) → A/B test (tác động kinh doanh, nhân quả).
- Router băm ổn định theo người dùng, salt riêng cho từng thí nghiệm, bucket lồng nhau khi tăng tỉ lệ.
- Cỡ mẫu tỉ lệ nghịch bình phương hiệu ứng — quyết định trước; không nhìn trộm (hoặc dùng kiểm định tuần tự); bandit để tối ưu, A/B để kết luận.
- Hành động của mô hình thay đổi dữ liệu tương lai — giữ nhóm đối chứng, log hành động, nếu không mô hình sẽ tự làm hỏng mình.

## 📚 Tài liệu tham khảo

- Kohavi, Tang & Xu (2020) — *Trustworthy Online Controlled Experiments: A Practical Guide to A/B Testing*
- Johari et al. (2017) — *Peeking at A/B Tests: Why it matters, and what to do about it*
- Russo et al. (2018) — *A Tutorial on Thompson Sampling*
- Sculley et al. (2015) — *Hidden Technical Debt in Machine Learning Systems* (feedback loops)
- Lakkaraju et al. (2017) — *The Selective Labels Problem*
- Google SRE Workbook — *Canarying Releases*